In [1]:
import os
import random
import shutil
from pathlib import Path

# ==========================
# Paths
# ==========================
SOURCE = Path("/kaggle/input/datasets/fatmaeissa/sorting-system-yolov8/train")

IMG_SRC = SOURCE / "images"
LBL_SRC = SOURCE / "labels"

DEST = Path("/kaggle/working/orange_dataset")

random.seed(42)

# ==========================
# Create folders
# ==========================

for split in ["train", "valid", "test"]:
    (DEST / split / "images").mkdir(parents=True, exist_ok=True)
    (DEST / split / "labels").mkdir(parents=True, exist_ok=True)

# ==========================
# Read images
# ==========================

images = []

for ext in ["*.jpg", "*.jpeg", "*.png"]:
    images.extend(list(IMG_SRC.glob(ext)))

print(f"Total Images = {len(images)}")

random.shuffle(images)

# ==========================
# Split
# ==========================

n = len(images)

train_size = int(n * 0.8)
valid_size = int(n * 0.1)

train_imgs = images[:train_size]
valid_imgs = images[train_size:train_size+valid_size]
test_imgs = images[train_size+valid_size:]

print(f"Train : {len(train_imgs)}")
print(f"Valid : {len(valid_imgs)}")
print(f"Test  : {len(test_imgs)}")

# ==========================
# Copy function
# ==========================

def copy_files(img_list, split):

    for img in img_list:

        shutil.copy(img, DEST / split / "images" / img.name)

        label = LBL_SRC / (img.stem + ".txt")

        if label.exists():
            shutil.copy(label, DEST / split / "labels" / label.name)

# ==========================
# Copy
# ==========================

copy_files(train_imgs, "train")
copy_files(valid_imgs, "valid")
copy_files(test_imgs, "test")

print("Dataset Split Done!")

Total Images = 710
Train : 568
Valid : 71
Test  : 71
Dataset Split Done!


In [2]:
from pathlib import Path

yaml_text = f"""
path: {Path('/kaggle/working/orange_dataset')}

train: train/images
val: valid/images
test: test/images

names:
  0: orange
"""

with open("/kaggle/working/orange_dataset/data.yaml","w") as f:
    f.write(yaml_text)

print("data.yaml Created!")

data.yaml Created!


In [3]:
import os

for split in ["train","valid","test"]:

    imgs = len(os.listdir(f"/kaggle/working/orange_dataset/{split}/images"))
    lbls = len(os.listdir(f"/kaggle/working/orange_dataset/{split}/labels"))

    print(split)
    print("Images :", imgs)
    print("Labels :", lbls)
    print("-"*30)

train
Images : 568
Labels : 568
------------------------------
valid
Images : 71
Labels : 71
------------------------------
test
Images : 71
Labels : 71
------------------------------


In [4]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.7 MB/s eta 0:00:00a 0:00:01


In [5]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
model = YOLO("yolo11n.pt")

In [7]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="/kaggle/working/orange_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    workers=2,
    patience=5,
    cache=True,
    pretrained=True,
    project="OrangeDetection",
    name="YOLO11",
    exist_ok=True,
    plots=True,
    seed=42
)

Ultralytics 8.4.91 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/orange_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLO11, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overl